Author: Krish

In [218]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from datetime import date

In [219]:
data_path = "/Users/viviadams/Downloads/Final_CAR_data"
#adhoc_data_path = 'C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Fall 2025\\LegalAid\\Adhoc\\Adhoc datasets\\'

In [220]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
df_main

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason


In [221]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)

1 CAR - EP, Flow, Activity, Queue, & Agent Names (01-12-25 - 01-18-25) (52010, 7)
2 CAR - EP, Flow, Activity, Queue, & Agent Names (01-19-25 - 02-01-25) (95495, 7)
3 CAR - EP, Flow, Activity, Queue, & Agent Names (02-02-25 - 02-15-25) (90056, 7)
4 CAR - EP, Flow, Activity, Queue, & Agent Names (02-16-25 - 03-01-25) (88186, 7)
5 CAR - EP, Flow, Activity, Queue, & Agent Names (03-02-25 - 03-15-25) (86377, 7)
6 CAR - EP, Flow, Activity, Queue, & Agent Names (04-07-24 - 04-20-24) (88766, 7)
7 CAR - EP, Flow, Activity, Queue, & Agent Names (04-21-24 - 05-04-24) (89643, 7)
8 CAR - EP, Flow, Activity, Queue, & Agent Names (05-05-24 - 05-18-24) (82575, 7)
9 CAR - EP, Flow, Activity, Queue, & Agent Names (05-19-24 - 06-01-24) (71103, 7)
10 CAR - EP, Flow, Activity, Queue, & Agent Names (06-02-24 - 06-15-24) (84354, 7)
11 CAR - EP, Flow, Activity, Queue, & Agent Names (06-16-24 - 06-29-24) (82124, 7)
12 CAR - EP, Flow, Activity, Queue, & Agent Names (06-30-24 - 07-13-24) (79752, 7)
13 CAR - EP, 

In [222]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour
df_main.sort_values(by = ['Contact Session ID', 'Activity Start Timestamp'], inplace=True)
df_main.reset_index(inplace = True, drop = True)

In [223]:
def custdata(id):
    return df_main.loc[df_main['Contact Session ID'] == id,:]
def act_rows(act):
    return df_main.loc[df_main['Activity Name'] == act,:]
def act_data(act):
    get_act_rows = act_rows(act)
    return df_main.loc[df_main['Contact Session ID'].isin(get_act_rows['Contact Session ID']),:]
def data_rows(act, data):
    return data.loc[data['Activity Name'] == act,:]
def col_data(col, val):
    get_col_rows = df_main.loc[df_main[col] == val,:]
    return df_main.loc[df_main['Contact Session ID'].isin(get_col_rows['Contact Session ID']),:]
def col_rows(col, val):
    return df_main.loc[df_main[col] == val,:]
def calls_for_activity(act_name):
    return df_main.loc[df_main['Activity Name'] == act_name,'Contact Session ID'].nunique()
def tagging(data):     
    ids_already_present = data.loc[data['Contact Session ID'].isin(all_data['Contact Session ID']),'Contact Session ID'].nunique()
    print("IDs already present in the previously tagged data = ", ids_already_present)
    data = data.loc[~data['Contact Session ID'].isin(all_data['Contact Session ID']),:]
    print("IDs added = ", data['Contact Session ID'].nunique())    
    print("% IDs added = ", round(100*data['Contact Session ID'].nunique()/248422),"%")
    print("IDs per weekday = ", round(data['Contact Session ID'].nunique()/370))
    appended_data = pd.concat([all_data, data])
    print("IDs in tagged data = ", appended_data['Contact Session ID'].nunique())
    print("% data tagged = ", round(100*appended_data['Contact Session ID'].nunique()/248422),"%")
    return appended_data

In [224]:
df_main_copy = df_main.copy()
df_main_copy = df_main_copy.loc[~df_main['Activity Name'].isna(),:]

#### Main-menu only

In [225]:
num_EPs = df_main.groupby('Contact Session ID')['EP Name'].nunique()
one_ep_data = df_main.loc[df_main['Contact Session ID'].isin(num_EPs[num_EPs == 1].index),:]
main_menu_rows = one_ep_data.loc[one_ep_data['EP Name'] == 'Main Number Telephony EP',:]
main_menu_data = df_main.loc[df_main['Contact Session ID'].isin(main_menu_rows['Contact Session ID']),:]

num_acts = df_main.groupby('Contact Session ID')['Activity Name'].nunique()
no_acts_rem = df_main.loc[(df_main['Contact Session ID'].isin(num_acts[(num_acts == 0)].index)) & \
(df_main['EP Name'] == 'Pre-Legal Menu Seniors Menu Telephony EP'),:]
no_acts_rem_data = df_main.loc[df_main['Contact Session ID'].isin(no_acts_rem['Contact Session ID']),:]

main_menu_data = pd.concat([main_menu_data, no_acts_rem_data])
main_menu_data['caller_type'] = 'Main menu only'
print("% data = ", round(100*98352/248422),"%")
print("calls per day = ", round(98352/370))
print("calls per day = ",main_menu_data['Contact Session ID'].nunique())

% data =  40 %
calls per day =  266
calls per day =  98357


## closed hours

In [226]:
closed_hours_data = col_data('EP Name', 'Closed Hours-Holidays Menu Telephony EP')
closed_hours_data['caller_type'] = 'Closed hours callers'
print(closed_hours_data['Contact Session ID'].nunique())

ids1 = set(main_menu_data["Contact Session ID"])
ids2 = set(closed_hours_data["Contact Session ID"])
common_12 = ids1 & ids2
if len(common_12) == 0:
    print("No overlap with the previously tagged data")

all_data = pd.concat([main_menu_data,closed_hours_data])
print("% data = ", round(100*closed_hours_data['Contact Session ID'].nunique()/248422),"%")
print("calls per day = ", round(closed_hours_data['Contact Session ID'].nunique()/370))
all_data['Contact Session ID'].nunique()

21861
No overlap with the previously tagged data
% data =  9 %
calls per day =  59


120218

### Farmworker data
calls already tagged as during closed hours excluded
Removing farmworker data from analysis due to high abandonment rate

In [227]:
farm_data = act_data('FarmworkerMainMenu')
farmworker_rows = one_ep_data.loc[one_ep_data['EP Name'] == 'Farmworker Main Number Telephony EP',:]
farmworker_data = df_main.loc[df_main['Contact Session ID'].isin(farmworker_rows['Contact Session ID']),:]
more_farm_data = farmworker_data.loc[~farmworker_data['Contact Session ID'].isin(farm_data['Contact Session ID']),:]
farm_data = pd.concat([farm_data, more_farm_data])
farm_data['caller_type'] = 'Farmworker'
all_data = tagging(farm_data)

IDs already present in the previously tagged data =  90
IDs added =  10310
% IDs added =  4 %
IDs per weekday =  28
IDs in tagged data =  130528
% data tagged =  53 %


### Non-seniors legal menu
Calls that reach LegalMenu2, but never reach SuburbsOrCityMenu

In [228]:
legmen2_data = act_data('LegalMenu1')
leg_senior_data_rows = data_rows('SuburbsOrCityMenu',legmen2_data)
#Non-seniors never reach the suburbs or city menu
leg_non_senior_data = legmen2_data.loc[~legmen2_data['Contact Session ID'].isin(leg_senior_data_rows['Contact Session ID']),:]
leg_non_senior_data['caller_type'] = 'Non-senior legal issue'
all_data = tagging(leg_non_senior_data)

IDs already present in the previously tagged data =  581
IDs added =  79085
% IDs added =  32 %
IDs per weekday =  214
IDs in tagged data =  209613
% data tagged =  84 %


## Seniors
### Suburban seniors
Calls that reach SuburbanSeniorsMenu menu

In [229]:
suburban_seniors_data = act_data('SuburbanSeniorsMenu')
suburban_seniors_data['caller_type'] = 'Suburban senior'
all_data = tagging(suburban_seniors_data)

IDs already present in the previously tagged data =  14
IDs added =  8305
% IDs added =  3 %
IDs per weekday =  22
IDs in tagged data =  217918
% data tagged =  88 %


### City Seniors

In [230]:
city_seniors = act_data('SeniorsADAPTMenu')
city_seniors['caller_type'] = 'Chicago senior'
all_data = tagging(city_seniors)

IDs already present in the previously tagged data =  55
IDs added =  17975
% IDs added =  7 %
IDs per weekday =  49
IDs in tagged data =  235893
% data tagged =  95 %


### Non cook county seniors

In [231]:
noncook_seniors = act_data('SeniorNotCookCoMenu')
noncook_seniors['caller_type'] = 'Non-Cook County Senior'
all_data = tagging(noncook_seniors)

IDs already present in the previously tagged data =  10
IDs added =  1302
% IDs added =  1 %
IDs per weekday =  4
IDs in tagged data =  237195
% data tagged =  95 %



### Unclassified seniors

In [232]:
#unclassified_seniors = act_data('SuburbsOrCityMenu')
#unclassified_seniors['caller_type'] = 'Unclassified seniors'
#all_data = tagging(unclassified_seniors)

All these seniors abandoned call before making a choice

## Intake outdial EP

In [233]:
intake_rows = one_ep_data.loc[one_ep_data['EP Name'] == 'Intake Outdial EP',:]
intake_outdial_data = df_main.loc[df_main['Contact Session ID'].isin(intake_rows['Contact Session ID']),:]
intake_outdial_data['caller_type'] = 'Intake Outdial'
all_data = tagging(intake_outdial_data)

IDs already present in the previously tagged data =  0
IDs added =  9395
% IDs added =  4 %
IDs per weekday =  25
IDs in tagged data =  246590
% data tagged =  99 %


### Rest

In [234]:
rest_data = df_main.loc[~df_main['Contact Session ID'].isin(all_data['Contact Session ID']),:]

In [235]:
rest_data_nomissact = rest_data.loc[~rest_data['Activity Name'].isna(),:]
last_act = rest_data_nomissact.groupby('Contact Session ID').tail(1)[['Contact Session ID', 'Activity Name']]
rest_data_nomissact.groupby('Contact Session ID').tail(1)[['Activity Name']].value_counts()

Activity Name          
SeniorsMenu                1197
SuburbsOrCityMenu           443
SeniorsConfirmationMenu     211
MainMenu                      1
Name: count, dtype: int64

All these calls are abandoned before information about the caller was obtained

In [236]:
rest_data['caller_type'] = 'unclassified (with access to legal issues menu)'
all_data = tagging(rest_data)

IDs already present in the previously tagged data =  0
IDs added =  1852
% IDs added =  1 %
IDs per weekday =  5
IDs in tagged data =  248442
% data tagged =  100 %


### caller type summary

In [237]:
call_type = pd.DataFrame({'calls':all_data.groupby('caller_type')['Contact Session ID'].nunique().sort_values(ascending = False)})
call_type['percent_calls'] = round(100*call_type['calls'] / 248442)
call_type['calls_per_weekday'] = round(call_type['calls'] / 370)
call_type

,calls,percent_calls,calls_per_weekday
caller_type,,,
Main menu only,98357,40.0,266.0
Non-senior legal issue,79085,32.0,214.0
Closed hours callers,21861,9.0,59.0
Chicago senior,17975,7.0,49.0
Farmworker,10310,4.0,28.0
Intake Outdial,9395,4.0,25.0
Suburban senior,8305,3.0,22.0
unclassified (with access to legal issues menu),1852,1.0,5.0
Non-Cook County Senior,1302,1.0,4.0


In [238]:
all_data['Overall_tag'] = "Legal issue"
all_data.loc[all_data['caller_type'].isin(['Farmworker','Intake Outdial','Closed hours callers']),'Overall_tag'] = 'Other'
all_data.loc[all_data['caller_type'] == 'Main menu only','Overall_tag'] = 'Main menu'
call_type = pd.DataFrame({'calls':all_data.groupby('Overall_tag')['Contact Session ID'].nunique().sort_values(ascending = False)})
call_type['percent_calls'] = round(100*call_type['calls'] / 248442)
call_type['calls_per_weekday'] = round(call_type['calls'] / 370)
call_type

,calls,percent_calls,calls_per_weekday
Overall_tag,,,
Legal issue,108519,44.0,293.0
Main menu,98357,40.0,266.0
Other,41566,17.0,112.0


In [239]:
# Filter to non-null Activity Name
all_data_non_null = all_data[all_data['Activity Name'].notna()]

# Get the row with the latest timestamp per Contact Session ID
last_activity = (
    all_data_non_null
    .sort_values('Activity Start Timestamp')
    .groupby('Contact Session ID')
    .tail(1)
)

last_activity = last_activity.rename(columns={'Activity Name' : 'Final Activity'})
last_activity_rele = last_activity.loc[:, ['Contact Session ID', 'Final Activity']]
all_act = pd.merge(all_data, last_activity_rele, how = 'outer')


In [254]:
def check_last_act(df, activity_col="Final Activity", outcome_col="Outcome", subtype_col = 'Subtype', rules1=None, rules2 = None, default="Unknown"):
    
    df[outcome_col] = df[activity_col].map(rules1).fillna(default)
    df[subtype_col] = df[activity_col].map(rules2).fillna(default)

    return df



outcome_rules = {
'ClosedQueueMenu' : 'Negative', 
'StaffDirectoryEnglishTransfer': 'Positive', 
'LanguageSelectionMenu': 'Negative',
'ClinicVoiceMailTransfer': 'Positive', 
'MainMenu': 'Negative', 
'DisconnectContact1': 'Negative', 
'LegalMenu2': 'Negative', 
'FarmworkerMainMenu': 'Negative', 
'SetCallerID': 'Positive', 
'LegalServerScreenPop': 'Positive', 
'FrontDeskTransfer1': 'Neutral', 
'ClosedMenu': 'Neutral', 
'FrontDeskTransfer2': 'Positive', 
'OtherLegalMenu': 'Neutral', 
'StaffDirectorySpanishTransfer' : 'Positive', 
'FrontDeskTransfer' : 'Neutral', 
'TenantMenu' : 'Negative', 
'CriminalRecordsVoicemailTransfer' : 'Positive', 
'OtherLegalOtherMenu' : 'Neutral', 
'DivorceOrParentingMenu' : 'Negative', 
'QueueMenu1' : 'Negative', 
'LegalMenu1' : 'Negative', 
'SeniorsMenu' : 'Negative', 
'TenantDeterrenceMenu' : 'Neutral', 
'HousingMenu' : 'Negative', 
'HIVVoicemailTransfer' : 'Positive', 
'PlayMOH300s' : 'Negative', 
'HIVMenu' : 'Negative', 
'ScreenPopProcessComplete' : 'Positive', 
'AddressFaxHoursMenu' : 'Positive', 
'PreTenantMenu' : 'Neutral', 
'FamilyMenu' : 'Negative', 
'FrontDeskTransfer3' : 'Neutral', 
'ChildSupportMenu' : 'Neutral', 
'ImmigrationMenu' : 'Negative', 
'SeniorsADAPTMenu' : 'Negative',
'OtherLegalPersonalInjuryMenu' : 'Neutral', 
'OtherLegalCriminalCaseMenu' : 'Neutral',
'SuburbsOrCityMenu' : 'Negative', 
'SuburbanSeniorsMenu' : 'Negative', 
'TransferToSafeHaven' : 'Positive', 
'SimpleDivorceMenu' : 'Neutral', 
'BenefitsMenu' : 'Negative', 
'SubSeniorPreQueueMessage1_1': 'Negative', 
'ThankYouGoodbye' : 'Neutral', 
'IntakePreQueueMessage1' : 'Negative',
'SeniorsConfirmationMenu' : 'Negative', 
'VeteransBenefitsVoicemailTransfer' : 'Positive', 
'EmploymentMenu' : 'Negative',
'No activity' : 'Negative',
'DisconnectContact2' : 'Neutral', 
'SubSeniorConsumerQueue' : 'Negative',
'SubSeniorHomeownerQueue' : 'Negative',
'PlayCCBConfirmation' : 'Negative', 
'EducationSPQueue' : 'Negative', 
'SubSeniorTenantQueue' : 'Negative' , 
'ImmigrationSPQueue' : 'Negative', 
'SubSeniorOtherQueue' : 'Negative', 
'SubSeniorPreQueueMessage1_2' : 'Negative', 
'ConfirmCallbackNumber' : 'Negative', 
'CollectCallbackNumber' : 'Negative', 
'CallbackRetry' : 'Negative', 
'PlayErrorMessage' : 'Negative', 
'ReadANI' : 'Negative', 
'GetLoggedInSubSeniorTenantAgents' : 'Negative', 
'WorkersCompMenu' : 'Neutral', 
'DisconnectContact' : 'Negative', 
'HelpWithLegalorOtherReasonMenu' : 'Negative', 
'PreQueueMessage2' : 'Negative', 
'ComplimentOrComplaintMenu' : 'Positive', 
'MigrantVoicemailTransfer' : 'Positive', 
'DisconnectCallbackContact' : 'Negative', 
'HolidayPrompt' : 'Neutral', 
'ImmigrationOtherMenu' : 'Neutral', 
'SeniorNotCookCoMenu' : 'Neutral', 
'AppointmentMenu' : 'Negative', 
'TraffickingVoicemailTransfer' : 'Positive', 
'GetLoggedInSubSeniorTenantSPAgents': 'Negative', 
'GetLoggedInOtherSubSeniorsAgents' : 'Negative', 
'GetLoggedInConsumerAgents': 'Negative', 
'GetLoggedInSubSeniorOtherSPAgents': 'Negative', 
'GetLoggedInSubSeniorOtherAgents' : 'Negative'
}

subtype_rules = {
'ClosedQueueMenu' : 'Closed Queue',
'StaffDirectoryEnglishTransfer' : 'Staff Directory', 
'LanguageSelectionMenu' : 'Abandoned', 
'MainMenu' : 'Abandoned', 
'LegalMenu2' : 'Abandoned',
'FarmworkerMainMenu' : 'Abandoned', 
'SetCallerID' : 'Connected to agent', 
'LegalServerScreenPop' : 'Connected to agent', 
'FrontDeskTransfer1' : 'No language selected (FD1)', 
'ClosedMenu' : 'Closed Hours', 
'FrontDeskTransfer2' : 'Feedback', 
'StaffDirectorySpanishTransfer' : 'Staff Directory', 
'FrontDeskTransfer' : 'Interpreter needed (FD, Legal menu)', 
'TenantMenu' : 'Abandoned', 
'CriminalRecordsVoicemailTransfer' : 'Reached Voicemail',
'OtherLegalOtherMenu' : 'LAC does not serve',
'DivorceOrParentingMenu' : 'Abandoned',
'QueueMenu1' : 'Abandoned', 
'LegalMenu1' : 'Abandoned', 
'SeniorsMenu' : 'Abandoned', 
'TenantDeterrenceMenu' : 'LAC does not serve', 
'HousingMenu' : 'Abandoned', 
'HIVVoicemailTransfer' : 'Reached Voicemail', 
'PlayMOH300s' : 'Abandoned', 
'HIVMenu' : 'Abandoned', 
'ScreenPopProcessComplete' : 'Connected to agent', 
'AddressFaxHoursMenu' : 'Reached Voicemail', 
'PreTenantMenu' : 'LAC does not serve', 
'FamilyMenu' : 'Abandoned',
'FrontDeskTransfer3' : 'No selection on Main menu (FD3)', 
'ChildSupportMenu' : 'LAC does not serve', 
'ImmigrationMenu' : 'Abandoned', 
'SeniorsADAPTMenu' : 'Abandoned', 
'OtherLegalPersonalInjuryMenu' : 'LAC does not serve', 
'OtherLegalCriminalCaseMenu' : 'LAC does not serve', 
'SuburbsOrCityMenu' : 'Abandoned', 
'SuburbanSeniorsMenu' : 'Abandoned', 
'TransferToSafeHaven' : 'Chatbot experiment (Family menu)', 
'SimpleDivorceMenu' : 'LAC does not serve', 
'BenefitsMenu' : 'Abandoned', 
'SubSeniorPreQueueMessage1_1': 'Abandoned', 
'ThankYouGoodbye' : 'LAC does not serve', 
'IntakePreQueueMessage1' : 'Abandoned', 
'SeniorsConfirmationMenu' : 'Abandoned', 
'VeteransBenefitsVoicemailTransfer' : 'Reached Voicemail', 
'EmploymentMenu' : 'Abandoned', 
'No activity' : 'Abandoned', 
'DisconnectContact2' : 'LAC does not serve', 
'SubSeniorConsumerQueue' : 'Abandoned', 
'SubSeniorHomeownerQueue' : 'Abandoned',
'PlayCCBConfirmation' : 'Failed Courtesy Callback', 
'EducationSPQueue' : 'Abandoned', 
'SubSeniorTenantQueue' : 'Abandoned', 
'ImmigrationSPQueue' : 'Abandoned', 
'SubSeniorOtherQueue' : 'Abandoned', 
'SubSeniorPreQueueMessage1_2' : 'Abandoned', 
'ConfirmCallbackNumber' : 'Abandoned', 
'CollectCallbackNumber' : 'Abandoned',
'CallbackRetry' : 'Failed Courtesy Callback', 
'PlayErrorMessage' : 'Abandoned', 
'ReadANI' : 'Failed Courtesy Callback', 
'GetLoggedInSubSeniorTenantAgents' : 'Abandoned', 
'WorkersCompMenu' : 'LAC does not serve', 
'DisconnectContact' : 'Outbound Call Failed', 
'HelpWithLegalorOtherReasonMenu' : 'Abandoned', 
'PreQueueMessage2' : 'Abandoned', 
'ComplimentOrComplaintMenu' : 'Feedback', 
'MigrantVoicemailTransfer' : 'Reached Voicemail', 
'DisconnectCallbackContact' : 'Failed Courtesy Callback', 
'HolidayPrompt' : 'Closed Hours', 
'ImmigrationOtherMenu' : 'LAC does not serve', 
'SeniorNotCookCoMenu' : 'LAC does not serve', 
'AppointmentMenu' : 'Abandoned', 
'TraffickingVoicemailTransfer' : 'Reached Voicemail', 
'GetLoggedInSubSeniorTenantSPAgents': 'Abandoned', 
'GetLoggedInOtherSubSeniorsAgents' : 'Abandoned', 
'GetLoggedInConsumerAgents': 'Abandoned', 
'GetLoggedInSubSeniorOtherSPAgents': 'Abandoned', 
'GetLoggedInSubSeniorOtherAgents' : 'Abandoned'

}


check_last_act(all_act, rules1=outcome_rules, rules2=subtype_rules)

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour,caller_type,Overall_tag,Final Activity,Outcome,Subtype,Touched_Legal_Menu
0,00002422-f51f-458b-82d6-cfa5a3f36fd9,Main Number Telephony EP,NaN,NaN,2025-03-13 12:51:21,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
1,00002422-f51f-458b-82d6-cfa5a3f36fd9,NaN,LACMain,NaN,2025-03-13 12:51:21,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
2,00002422-f51f-458b-82d6-cfa5a3f36fd9,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-03-13 12:51:21,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
3,00002422-f51f-458b-82d6-cfa5a3f36fd9,Main Number Telephony EP,LACMain,NaN,2025-03-13 12:51:21,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
4,00002422-f51f-458b-82d6-cfa5a3f36fd9,Main Number Telephony EP,NaN,MainMenu,2025-03-13 12:51:31,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3328621,ffffa50d-6d02-4281-9141-087a806a8343,NaN,LACMain,NaN,2024-09-25 04:08:13,NaN,NaN,NaN,4,Main menu only,Main menu,LanguageSelectionMenu,Negative,Abandoned,False
3328622,ffffa50d-6d02-4281-9141-087a806a8343,Main Number Telephony EP,NaN,LanguageSelectionMenu,2024-09-25 04:08:13,NaN,NaN,NaN,4,Main menu only,Main menu,LanguageSelectionMenu,Negative,Abandoned,False
3328623,ffffa50d-6d02-4281-9141-087a806a8343,Main Number Telephony EP,LACMain,NaN,2024-09-25 04:08:13,NaN,NaN,NaN,4,Main menu only,Main menu,LanguageSelectionMenu,Negative,Abandoned,False
3328624,ffffa50d-6d02-4281-9141-087a806a8343,Main Number Telephony EP,NaN,LanguageSelectionMenu,2024-09-25 04:08:15,NaN,NaN,NaN,4,Main menu only,Main menu,LanguageSelectionMenu,Negative,Abandoned,False


In [257]:
menus_of_interest = [
    'OtherLegalPersonalInjuryMenu',
    'OtherLegalCriminalCaseMenu',
    'OtherLegalOtherMenu',
    'ClinicVoicemailTransfer'
]
# Define conditions
cond1 = (all_act['Final Activity'] == 'ClinicVoiceMailTransfer') & \
        (all_act['EP Name'] == 'Closed Queue Menu Telephony EP')
cond2 = (all_act['Final Activity'] == 'ClinicVoiceMailTransfer') & \
        (all_act['EP Name'] == 'Closed Hours-Holidays Menu Telephony EP')
cond3 = (all_act['Final Activity'] == 'DisconnectContact1') & \
        (all_act['EP Name'] == 'Closed Hours-Holidays Menu Telephony EP')
cond4 = (all_act['Final Activity'] == 'DisconnectContact1') & \
        (all_act['EP Name'] == 'Closed Queue Menu Telephony EP')
cond5 = (all_act['Final Activity'] == 'DisconnectContact1') & \
        (all_act['EP Name'] == 'Courtesy Callback Telephony EP')
cond6 = (all_act['Final Activity'] == 'DisconnectContact1') & \
        (all_act['EP Name'] == 'Pre-Legal Menu Seniors Menu Telephony EP')
cond7 = all_act['Touched_Legal_Menu']

conditions = [cond1, cond2, cond3, cond4, cond5, cond6, cond7]
outcome_choices = ['Negative', 'Neutral', 'Neutral', 'Negative', 'Negative', 'Neutral', 'Neutral']
subtype_choices = ['Closed Queue', 'Closed Hours', 'Closed Hours', 'Closed Queue', 
                   'Failed Courtesy Callback', 'LAC does not serve', 'LAC does not serve']




In [258]:
def assign_outcomes(df, 
                    activity_col="Final Activity",
                    ep_col="EP Name",
                    outcome_col="Outcome",
                    subtype_col="Subtype",
                    rules_outcome=None,   # dict mapping activity -> Outcome
                    rules_subtype=None,   # dict mapping activity -> Subtype
                    conditions=None,      # list of boolean conditions (for np.select)
                    cond_outcomes=None,   # list of Outcome values corresponding to conditions
                    cond_subtypes=None,   # list of Subtype values corresponding to conditions
                    default_outcome="Unknown",
                    default_subtype="Unknown"):
    """
    Vectorized assignment of Outcome and Subtype using:
    1. Explicit activity mappings (rules_outcome / rules_subtype)
    2. Conditional logic (conditions / np.select)
    """

    # Step 1: Apply conditional rules first (np.select)
    if conditions and cond_outcomes and cond_subtypes:
        df[outcome_col] = np.select(conditions, cond_outcomes, default=default_outcome)
        df[subtype_col] = np.select(conditions, cond_subtypes, default=default_subtype)
    else:
        df[outcome_col] = default_outcome
        df[subtype_col] = default_subtype

    # Step 2: Apply explicit mapping rules, but only for matching rows
    if rules_outcome:
        mask_outcome = df[activity_col].isin(rules_outcome.keys())
        df.loc[mask_outcome, outcome_col] = df.loc[mask_outcome, activity_col].map(rules_outcome)

    if rules_subtype:
        mask_subtype = df[activity_col].isin(rules_subtype.keys())
        df.loc[mask_subtype, subtype_col] = df.loc[mask_subtype, activity_col].map(rules_subtype)

    return df


assign_outcomes(all_act,rules_outcome=outcome_rules,
                     rules_subtype=subtype_rules,
                     conditions=conditions,
                     cond_outcomes=outcome_choices,
                     cond_subtypes=subtype_choices,
                     default_outcome='Unknown',
                     default_subtype='Unknown')

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour,caller_type,Overall_tag,Final Activity,Outcome,Subtype,Touched_Legal_Menu
0,00002422-f51f-458b-82d6-cfa5a3f36fd9,Main Number Telephony EP,NaN,NaN,2025-03-13 12:51:21,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
1,00002422-f51f-458b-82d6-cfa5a3f36fd9,NaN,LACMain,NaN,2025-03-13 12:51:21,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
2,00002422-f51f-458b-82d6-cfa5a3f36fd9,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-03-13 12:51:21,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
3,00002422-f51f-458b-82d6-cfa5a3f36fd9,Main Number Telephony EP,LACMain,NaN,2025-03-13 12:51:21,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
4,00002422-f51f-458b-82d6-cfa5a3f36fd9,Main Number Telephony EP,NaN,MainMenu,2025-03-13 12:51:31,NaN,NaN,NaN,12,Main menu only,Main menu,StaffDirectoryEnglishTransfer,Positive,Staff Directory,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3328621,ffffa50d-6d02-4281-9141-087a806a8343,NaN,LACMain,NaN,2024-09-25 04:08:13,NaN,NaN,NaN,4,Main menu only,Main menu,LanguageSelectionMenu,Negative,Abandoned,False
3328622,ffffa50d-6d02-4281-9141-087a806a8343,Main Number Telephony EP,NaN,LanguageSelectionMenu,2024-09-25 04:08:13,NaN,NaN,NaN,4,Main menu only,Main menu,LanguageSelectionMenu,Negative,Abandoned,False
3328623,ffffa50d-6d02-4281-9141-087a806a8343,Main Number Telephony EP,LACMain,NaN,2024-09-25 04:08:13,NaN,NaN,NaN,4,Main menu only,Main menu,LanguageSelectionMenu,Negative,Abandoned,False
3328624,ffffa50d-6d02-4281-9141-087a806a8343,Main Number Telephony EP,NaN,LanguageSelectionMenu,2024-09-25 04:08:15,NaN,NaN,NaN,4,Main menu only,Main menu,LanguageSelectionMenu,Negative,Abandoned,False


In [259]:
final_act = all_act.drop('Touched_Legal_Menu', axis=1)


In [260]:
all_data_non_null2 = final_act[final_act['EP Name'].notna()]

# Get the row with the latest timestamp per Contact Session ID
last_activity2 = (
    all_data_non_null2
    .sort_values('Activity Start Timestamp')
    .groupby('Contact Session ID')
    .tail(1)
)

last_activity2 = last_activity2.rename(columns={'EP Name' : 'Final Menu'})
last_activity_rele2 = last_activity2.loc[:, ['Contact Session ID', 'Final Menu']]
final_act = pd.merge(final_act, last_activity_rele2, how = 'outer')

In [261]:
def update_final_menu(df,
                      final_menu_col="Final Menu",
                      ep_col="EP Name",
                      id_col="Contact Session ID",
                      time_col="Activity Start Timestamp",
                      ignored_eps=None):
    if ignored_eps is None:
        ignored_eps = [
            'All LAC Queues Telephony EP',
            'Courtesy Callback Telephony EP',
            'Closed Queue Menu Telephony EP'
        ]

    # Sort chronologically within each contact session
    df = df.sort_values([id_col, time_col])

    # Filter rows to only keep valid EPs
    valid_eps = df[~df[ep_col].isin(ignored_eps)]

    # Find last valid EP per contact session
    last_ep = valid_eps.groupby(id_col)[ep_col].last()

    # Update Final Menu column for all rows using map
    df[final_menu_col] = df[id_col].map(last_ep)

    return df


final_act2 = update_final_menu(final_act)

In [270]:
final_act_unique = final_act2.drop_duplicates(subset='Contact Session ID', keep='last')


In [272]:
summary = (
    final_act_unique.groupby(['Overall_tag', 'caller_type'])['Outcome']
      .value_counts()  # count number of each outcome type
      .unstack(fill_value=0)  # turn Outcome types into columns
      .reset_index()
     
)
summary

Outcome,Overall_tag,caller_type,Negative,Neutral,Positive,Unknown
0,Legal issue,Chicago senior,10201,5995,1752,27
1,Legal issue,Non-Cook County Senior,748,502,50,2
2,Legal issue,Non-senior legal issue,51312,19445,8161,167
3,Legal issue,Suburban senior,3792,845,3536,132
4,Legal issue,unclassified (with access to legal issues menu),1852,0,0,0
5,Main menu,Main menu only,40687,21322,36177,171
6,Other,Closed hours callers,9513,9208,3140,0
7,Other,Farmworker,10103,104,101,2
8,Other,Intake Outdial,3070,0,6320,5


In [273]:
final_legal = final_act_unique.loc[final_act_unique['caller_type'] == 'Non-senior legal issue']


In [280]:
family_menu = final_legal.loc[final_legal['Final Menu'] == 'Legal Family Menu Telephony EP', :]

summary = family_menu.groupby(['Outcome', 'Subtype']).size().reset_index(name='Count').sort_values(by='Count', ascending= False)
summary

,Outcome,Subtype,Count
1,Negative,Closed Queue,13217
3,Neutral,LAC does not serve,3192
0,Negative,Abandoned,2037
5,Positive,Connected to agent,1804
4,Positive,Chatbot experiment (Family menu),391
6,Unknown,Unknown,94
2,Negative,Failed Courtesy Callback,27


In [275]:
final_legal['Final Menu'].unique()

array(['Legal Family Menu Telephony EP', 'Legal Menu Telephony EP',
       'Legal Housing Menu Telephony EP', 'Other Legal Menu Telephony EP',
       'Legal Employment Menu Telephony EP',
       'Legal Benefits Menu Telephony EP',
       'Legal Immigration Menu Telephony EP',
       'Legal HIV Menu Telephony EP', 'Main Number Telephony EP',
       'Pre-Legal Menu Seniors Menu Telephony EP'], dtype=object)